# Adam epoch study 

Trains the CA-PINN (4 x 40, tanh) with 2000, 5000 and 10000 Adam epochs, each followed by at most 10000 L-BFGS iterations, for seeds 42, 1234 and 2024. The finite-difference reference is computed once and reused. The notebook collects the relative L2 errors, the Adam and L-BFGS run times, the number of L-BFGS iterations and objective evaluations, and the loss histories.

In [ ]:
# ============================================================
# Imports and project directory
# ============================================================

import os
import gc
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_DIR = Path(".")  # repository root (relative paths)
os.chdir(PROJECT_DIR)

PINN_CANDIDATES = [
    PROJECT_DIR / "CA_PINN_Solver.ipynb"
]

FD_CANDIDATES = [
    PROJECT_DIR / "FD_Solver.ipynb"
]

PINN_NOTEBOOK = next(
    (path for path in PINN_CANDIDATES if path.exists()),
    None,
)
FD_NOTEBOOK = next(
    (path for path in FD_CANDIDATES if path.exists()),
    None,
)
VALIDATION_FILE = PROJECT_DIR / "validation_utils.py"

if PINN_NOTEBOOK is None:
    raise FileNotFoundError(
        "PINN iteration-study notebook was not found. "
        f"Checked: {[str(path) for path in PINN_CANDIDATES]}"
    )

if FD_NOTEBOOK is None:
    raise FileNotFoundError(
        "FD validation notebook was not found. "
        f"Checked: {[str(path) for path in FD_CANDIDATES]}"
    )

if not VALIDATION_FILE.exists():
    raise FileNotFoundError(
        f"Required utility file was not found: {VALIDATION_FILE}"
    )

from validation_utils import rel_l2

print("Working directory:", Path.cwd())
print("PINN notebook:", PINN_NOTEBOOK.name)
print("FD notebook:", FD_NOTEBOOK.name)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# ============================================================
# Reproducibility
# ============================================================

os.environ["TF_DETERMINISTIC_OPS"] = "1"

try:
    tf.config.experimental.enable_op_determinism()
except Exception as exc:
    print("Deterministic operations warning:", repr(exc))

def set_all_seeds(seed: int) -> None:
    tf.keras.backend.clear_session()
    gc.collect()

    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
# ============================================================
# Study configuration
# ============================================================

# Keep True for the first one-run technical check.
# Change to False for the complete 9-run experiment.
TEST_MODE = False

# False preserves completed cases and allows safe resumption.
# True deletes and retrains an existing case.
OVERWRITE_EXISTING = False

RUN_FD_REFERENCE = False
RUN_PINN_CASES = False

# Fixed physical case
BASE_P0 = 1.0e21
BASE_PHI = 0.10
BASE_WCONS = 100.0

# Fixed model selected from previous studies
FIXED_HIDDEN_LAYERS = 4
FIXED_NEURONS = 40
FIXED_ACTIVATION = "tanh"

# Adam is the only varied training hyperparameter.
FULL_ADAM_EPOCHS = [2000, 5000, 10000]

# L-BFGS remains fixed for every Adam case.
FIXED_LBFGS_MAX_ITER = 10000
RUN_LBFGS = True

FULL_SEEDS = [42, 1234, 2024]

if TEST_MODE:
    ADAM_EPOCHS = [2000]
    SEEDS = [42]
else:
    ADAM_EPOCHS = FULL_ADAM_EPOCHS
    SEEDS = FULL_SEEDS

OUTPUT_ROOT = (
    PROJECT_DIR
    / "results"
    / "adam_iterations"
)
FD_REFERENCE_DIR = OUTPUT_ROOT / "fd_reference"
RUNS_DIR = OUTPUT_ROOT / "runs"
FIGURES_DIR = OUTPUT_ROOT / "figures"

for directory in [
    OUTPUT_ROOT,
    FD_REFERENCE_DIR,
    RUNS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUTPUT_ROOT / "adam_iteration_all_runs.csv"
SUMMARY_CSV = OUTPUT_ROOT / "adam_iteration_mean_std.csv"
TABLE_CSV = OUTPUT_ROOT / "adam_iteration_publication_table.csv"

print("Test mode:", TEST_MODE)
print("Adam epochs:", ADAM_EPOCHS)
print("Seeds:", SEEDS)
print("Number of PINN runs:", len(ADAM_EPOCHS) * len(SEEDS))
print("Fixed L-BFGS maximum iterations:", FIXED_LBFGS_MAX_ITER)
print("Output root:", OUTPUT_ROOT)

In [ ]:
# ============================================================
# Helper functions
# ============================================================

REQUIRED_PINN_FILES = [
    "pinn_psi.csv",
    "pinn_n.csv",
    "pinn_I.csv",
    "loss_history.csv",
    "training_metadata.csv",
]

REQUIRED_FD_FILES = [
    "fd_psi.csv",
    "fd_n.csv",
    "fd_I.csv",
]

LEFTOVER_GLOBALS = [
    "prebias_model",
    "main_model",
    "model",
    "layers",
    "results",
    "X_0",
    "X_r",
    "X_a",
    "X_c",
    "u_a",
    "u_c",
    "X_r_pre",
    "X_0_pre",
    "X_aPre",
    "X_cPre",
    "lb",
    "ub",
    "lb_pre",
    "ub_pre",
    "lb_main",
    "ub_main",
    "u0_interp_dict",
    "u_0",
    "x_0",
    "X_test",
    "X_list",
    "T_list",
    "psi_pred",
    "n_pred",
    "I_pred",
]

def clear_solver_globals() -> None:
    namespace = globals()

    for name in LEFTOVER_GLOBALS:
        namespace.pop(name, None)

    tf.keras.backend.clear_session()
    gc.collect()

def require_files(directory: Path, filenames: list[str]) -> None:
    missing = [
        filename
        for filename in filenames
        if not (directory / filename).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing files in {directory}: {missing}"
        )

def copy_fd_reference(case_dir: Path) -> None:
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)

    for filename in REQUIRED_FD_FILES:
        shutil.copy2(
            FD_REFERENCE_DIR / filename,
            case_dir / filename,
        )

def parse_bool(value) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        return False

    return str(value).strip().lower() in {
        "true",
        "1",
        "yes",
        "y",
    }

def compute_case_metrics(case_dir: Path) -> dict:
    require_files(
        case_dir,
        REQUIRED_PINN_FILES + REQUIRED_FD_FILES,
    )

    pinn_psi = pd.read_csv(case_dir / "pinn_psi.csv")
    pinn_n = pd.read_csv(case_dir / "pinn_n.csv")
    pinn_I = pd.read_csv(case_dir / "pinn_I.csv")

    fd_psi = pd.read_csv(case_dir / "fd_psi.csv")
    fd_n = pd.read_csv(case_dir / "fd_n.csv")
    fd_I = pd.read_csv(case_dir / "fd_I.csv")

    comparisons = [
        ("psi", pinn_psi, fd_psi, "psi_V"),
        ("n", pinn_n, fd_n, "n_m3"),
        ("I", pinn_I, fd_I, "I_m3"),
    ]

    for name, pinn_df, fd_df, value_column in comparisons:
        required_columns = {"x_nm", value_column}

        if not required_columns.issubset(pinn_df.columns):
            raise ValueError(
                f"{name}: PINN file is missing required columns."
            )

        if not required_columns.issubset(fd_df.columns):
            raise ValueError(
                f"{name}: FD file is missing required columns."
            )

        if len(pinn_df) != len(fd_df):
            raise ValueError(
                f"{name}: PINN and FD grid lengths differ."
            )

        max_dx = np.max(
            np.abs(
                pinn_df["x_nm"].to_numpy()
                - fd_df["x_nm"].to_numpy()
            )
        )

        if max_dx > 1.0e-9:
            raise ValueError(
                f"{name}: PINN and FD grids differ. "
                f"Maximum difference = {max_dx:.3e} nm."
            )

    metrics = {
        "relL2_psi": float(
            rel_l2(
                pinn_psi["psi_V"].to_numpy(),
                fd_psi["psi_V"].to_numpy(),
            )
        ),
        "relL2_n": float(
            rel_l2(
                pinn_n["n_m3"].to_numpy(),
                fd_n["n_m3"].to_numpy(),
            )
        ),
        "relL2_I": float(
            rel_l2(
                pinn_I["I_m3"].to_numpy(),
                fd_I["I_m3"].to_numpy(),
            )
        ),
    }

    # One balanced scalar for optional ranking.
    metrics["aggregate_error"] = float(
        np.exp(
            np.mean(
                np.log(
                    np.maximum(
                        [
                            metrics["relL2_psi"],
                            metrics["relL2_n"],
                            metrics["relL2_I"],
                        ],
                        1.0e-300,
                    )
                )
            )
        )
    )

    loss_df = pd.read_csv(
        case_dir / "loss_history.csv"
    )
    metadata_df = pd.read_csv(
        case_dir / "training_metadata.csv"
    )

    if loss_df.empty:
        raise ValueError("Loss-history CSV is empty.")

    if metadata_df.empty:
        raise ValueError("Training-metadata CSV is empty.")

    required_loss_columns = {
        "phase",
        "step",
        "total_loss",
        "L_psi",
        "L_n",
        "L_I",
    }
    missing_loss_columns = (
        required_loss_columns - set(loss_df.columns)
    )

    if missing_loss_columns:
        raise ValueError(
            "Loss-history CSV is missing columns: "
            f"{sorted(missing_loss_columns)}"
        )

    phases = set(loss_df["phase"].astype(str).str.lower())

    if "adam" not in phases:
        raise ValueError(
            "Adam phase was not found in the loss-history CSV."
        )

    if RUN_LBFGS and "lbfgs" not in phases:
        raise ValueError(
            "L-BFGS phase was not found in the loss-history CSV."
        )

    metadata = metadata_df.iloc[0]

    metrics.update({
        "final_total_loss": float(
            loss_df["total_loss"].iloc[-1]
        ),
        "adam_runtime_s": float(metadata["adam_runtime_s"]),
        "lbfgs_runtime_s": float(metadata["lbfgs_runtime_s"]),
        "training_runtime_s": float(metadata["total_runtime_s"]),
        "lbfgs_converged": parse_bool(
            metadata["lbfgs_converged"]
        ),
        "lbfgs_failed": parse_bool(
            metadata["lbfgs_failed"]
        ),
        "lbfgs_iterations": int(
            metadata["lbfgs_iterations"]
        ),
        "lbfgs_objective_evaluations": int(
            metadata["lbfgs_objective_evaluations"]
        ),
        "lbfgs_final_objective": float(
            metadata["lbfgs_final_objective"]
        ),
    })

    return metrics

def count_trainable_parameters() -> int:
    if "prebias_model" not in globals():
        raise RuntimeError(
            "prebias_model is unavailable after PINN execution."
        )

    return int(
        sum(
            np.prod(variable.shape)
            for variable in prebias_model.model.trainable_variables
        )
    )

In [ ]:
# ============================================================
# Generate the common FD reference once
# ============================================================

if RUN_FD_REFERENCE:
    print("=" * 80)
    print("Generating the common FD reference")
    print("=" * 80)

    clear_solver_globals()

    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI
    globals()["OUTPUT_DIR"] = str(FD_REFERENCE_DIR)

    fd_start = time.time()

    get_ipython().run_line_magic(
        "run",
        f'-i "{FD_NOTEBOOK}"',
    )

    fd_runtime = time.time() - fd_start
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)

    print(f"FD reference completed in {fd_runtime:.1f} s.")
else:
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)
    print("Using existing FD reference:", FD_REFERENCE_DIR)

In [ ]:
# ============================================================
# Run one Adam-iteration case
# ============================================================

def run_adam_iteration_case(
    adam_epochs: int,
    seed: int,
) -> dict:
    tag = (
        f"adam{adam_epochs}"
        f"_lbfgs{FIXED_LBFGS_MAX_ITER}"
        f"_L{FIXED_HIDDEN_LAYERS}"
        f"_W{FIXED_NEURONS}"
        f"_{FIXED_ACTIVATION}"
        f"_seed{seed}"
    )

    case_dir = RUNS_DIR / tag

    if case_dir.exists() and OVERWRITE_EXISTING:
        shutil.rmtree(case_dir)

    case_dir.mkdir(parents=True, exist_ok=True)

    already_complete = all(
        (case_dir / filename).exists()
        for filename in (
            REQUIRED_PINN_FILES + REQUIRED_FD_FILES
        )
    )

    common_fields = {
        "adam_epochs": int(adam_epochs),
        "lbfgs_max_iterations": FIXED_LBFGS_MAX_ITER,
        "hidden_layers": FIXED_HIDDEN_LAYERS,
        "neurons": FIXED_NEURONS,
        "activation": FIXED_ACTIVATION,
        "seed": int(seed),
        "P0": BASE_P0,
        "phi": BASE_PHI,
        "W_cons": BASE_WCONS,
        "case_dir": str(case_dir),
    }

    if already_complete and not OVERWRITE_EXISTING:
        print(f"Reusing completed case: {tag}")
        metrics = compute_case_metrics(case_dir)

        return {
            **common_fields,
            **metrics,
            "status": "reused",
            "error": "",
        }

    print("\n" + "=" * 80)
    print(f"Running case: {tag}")
    print(f"Adam epochs: {adam_epochs}")
    print(
        "Fixed maximum L-BFGS iterations:",
        FIXED_LBFGS_MAX_ITER,
    )
    print(f"Seed: {seed}")
    print("=" * 80)

    clear_solver_globals()
    set_all_seeds(seed)

    # Fixed physical settings.
    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI
    globals()["W_cons"] = BASE_WCONS

    # Fixed architecture and activation.
    globals()["HIDDEN_LAYERS"] = FIXED_HIDDEN_LAYERS
    globals()["NEURONS"] = FIXED_NEURONS
    globals()["ACTIVATION"] = FIXED_ACTIVATION

    # Training configuration.
    globals()["NUM_EPOCH_ADAM"] = int(adam_epochs)
    globals()["NUM_EPOCH_LBFGS"] = int(
        FIXED_LBFGS_MAX_ITER
    )
    globals()["RUN_LBFGS"] = RUN_LBFGS

    # Every case writes only into its own directory.
    globals()["OUTPUT_DIR"] = str(case_dir)

    try:
        if not RUN_PINN_CASES:
            raise RuntimeError(
                "RUN_PINN_CASES is False and the case is incomplete."
            )

        get_ipython().run_line_magic(
            "run",
            f'-i "{PINN_NOTEBOOK}"',
        )

        require_files(case_dir, REQUIRED_PINN_FILES)
        copy_fd_reference(case_dir)

        metrics = compute_case_metrics(case_dir)
        parameter_count = count_trainable_parameters()

        row = {
            **common_fields,
            "trainable_parameters": parameter_count,
            **metrics,
            "status": "ok",
            "error": "",
        }

        print(
            f"[{tag}] aggregate relative error = "
            f"{metrics['aggregate_error']:.3e}"
        )
        print(
            f"[{tag}] actual L-BFGS iterations = "
            f"{metrics['lbfgs_iterations']}"
        )
        print(
            f"[{tag}] L-BFGS objective evaluations = "
            f"{metrics['lbfgs_objective_evaluations']}"
        )

        return row

    except Exception as exc:
        print(f"[{tag}] FAILED: {repr(exc)}")

        return {
            **common_fields,
            "status": "failed",
            "error": repr(exc),
        }

    finally:
        clear_solver_globals()

In [ ]:
# ============================================================
# Run or resume the sweep
# ============================================================

if RUNS_CSV.exists() and not OVERWRITE_EXISTING:
    existing_df = pd.read_csv(RUNS_CSV)
    rows = existing_df.to_dict("records")
else:
    rows = []

for adam_epochs in ADAM_EPOCHS:
    for seed in SEEDS:
        key = (int(adam_epochs), int(seed))

        matching_rows = [
            row
            for row in rows
            if (
                int(row.get("adam_epochs", -1)),
                int(row.get("seed", -1)),
            ) == key
        ]

        completed_record = any(
            str(row.get("status", "")).lower()
            in {"ok", "reused"}
            for row in matching_rows
        )

        if completed_record and not OVERWRITE_EXISTING:
            print("Already recorded, skipping:", key)
            continue

        # Remove an old failed or incomplete record before retrying.
        rows = [
            row
            for row in rows
            if (
                int(row.get("adam_epochs", -1)),
                int(row.get("seed", -1)),
            ) != key
        ]

        row = run_adam_iteration_case(
            adam_epochs=adam_epochs,
            seed=seed,
        )
        rows.append(row)

        pd.DataFrame(rows).to_csv(
            RUNS_CSV,
            index=False,
        )
        print("Updated:", RUNS_CSV)

print("\nAdam-iteration sweep completed.")

In [ ]:
import pandas as pd

df = pd.read_csv(RUNS_CSV)

df.loc[
    (df["adam_epochs"] == 2000) &
    (df["seed"] == 42),
    "case_dir"
] = str(
    RUNS_DIR /
    "adam2000_lbfgs10000_L4_W40_tanh_seed42"
)

df.to_csv(RUNS_CSV, index=False)

print("CSV updated successfully.")

In [ ]:
# ============================================================
# Inspect individual runs
# ============================================================

all_runs = pd.read_csv(RUNS_CSV)

requested_mask = (
    all_runs["adam_epochs"].isin(ADAM_EPOCHS)
    & all_runs["seed"].isin(SEEDS)
)

requested_runs = all_runs[requested_mask].copy()

display_columns = [
    "adam_epochs",
    "seed",
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "aggregate_error",
    "adam_runtime_s",
    "lbfgs_runtime_s",
    "training_runtime_s",
    "lbfgs_iterations",
    "lbfgs_objective_evaluations",
    "lbfgs_converged",
    "lbfgs_failed",
    "status",
]

display(
    requested_runs[
        [
            column
            for column in display_columns
            if column in requested_runs.columns
        ]
    ].sort_values(["adam_epochs", "seed"])
)

successful = requested_runs[
    requested_runs["status"]
    .astype(str)
    .str.lower()
    .isin({"ok", "reused"})
].copy()

expected_runs = len(ADAM_EPOCHS) * len(SEEDS)

print(
    "Successful requested runs:",
    len(successful),
    "/",
    expected_runs,
)

if len(successful) != expected_runs:
    print(
        "Warning: the requested sweep is incomplete. "
        "Review failed cases before interpreting the summary."
    )

In [ ]:
# ============================================================
# Mean and standard deviation over seeds
# ============================================================

if successful.empty:
    raise RuntimeError(
        "No successful runs are available for summarization."
    )

metrics_to_summarize = [
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "aggregate_error",
    "final_total_loss",
    "adam_runtime_s",
    "lbfgs_runtime_s",
    "training_runtime_s",
    "lbfgs_iterations",
    "lbfgs_objective_evaluations",
    "lbfgs_final_objective",
]

summary = (
    successful
    .groupby("adam_epochs", as_index=False)
    .agg(
        runs=("seed", "count"),
        **{
            f"{metric}_mean": (metric, "mean")
            for metric in metrics_to_summarize
        },
        **{
            f"{metric}_std": (metric, "std")
            for metric in metrics_to_summarize
        },
    )
    .sort_values("adam_epochs")
)

summary.to_csv(SUMMARY_CSV, index=False)

display(summary)
print("Saved:", SUMMARY_CSV)

In [ ]:
# ============================================================
# Paper-style phase-separated loss-history figure
# ============================================================

# One representative seed is used only for visualizing convergence.
# The numerical publication table is based on all seeds.
PLOT_SEED = SEEDS[0]

fig, axes = plt.subplots(
    len(ADAM_EPOCHS),
    2,
    figsize=(11.0, 3.5 * len(ADAM_EPOCHS)),
    squeeze=False,
)

loss_columns = [
    ("total_loss", "Total loss"),
    ("L_psi", r"$\mathcal{L}_{\psi}$"),
    ("L_n", r"$\mathcal{L}_{n}$"),
    ("L_I", r"$\mathcal{L}_{I}$"),
]

for row_index, adam_epochs in enumerate(ADAM_EPOCHS):

    # Check that this Adam/seed combination was completed successfully.
    match = successful[
        (successful["adam_epochs"] == adam_epochs)
        & (successful["seed"] == PLOT_SEED)
    ]

    if match.empty:
        print(
            f"No successful result found for "
            f"Adam={adam_epochs}, seed={PLOT_SEED}."
        )
        axes[row_index, 0].set_visible(False)
        axes[row_index, 1].set_visible(False)
        continue

    # Construct the case directory from the current RUNS_DIR.
    # This avoids using an outdated case_dir path stored in the dataframe.
    case_name = (
        f"adam{int(adam_epochs)}"
        f"_lbfgs{int(FIXED_LBFGS_MAX_ITER)}"
        f"_L{int(FIXED_HIDDEN_LAYERS)}"
        f"_W{int(FIXED_NEURONS)}"
        f"_{FIXED_ACTIVATION}"
        f"_seed{int(PLOT_SEED)}"
    )

    case_dir = RUNS_DIR / case_name
    loss_file = (
        case_dir / "loss_history.csv"
    )

    print("Reading:", loss_file)

    if not loss_file.exists():
        print(
            f"Warning: loss-history file was not found for "
            f"Adam={adam_epochs}, seed={PLOT_SEED}:\n"
            f"{loss_file}"
        )
        axes[row_index, 0].set_visible(False)
        axes[row_index, 1].set_visible(False)
        continue

    loss_df = pd.read_csv(loss_file)

    # Validate the columns required for plotting.
    required_columns = {
        "phase",
        "step",
        "total_loss",
        "L_psi",
        "L_n",
        "L_I",
    }

    missing_columns = required_columns.difference(
        loss_df.columns
    )

    if missing_columns:
        print(
            f"Warning: missing columns in {loss_file.name}: "
            f"{sorted(missing_columns)}"
        )
        axes[row_index, 0].set_visible(False)
        axes[row_index, 1].set_visible(False)
        continue

    loss_df["phase"] = (
        loss_df["phase"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    adam_df = loss_df[
        loss_df["phase"] == "adam"
    ].copy()

    lbfgs_df = loss_df[
        loss_df["phase"] == "lbfgs"
    ].copy()

    if adam_df.empty:
        print(
            f"Warning: no Adam loss history found for "
            f"Adam={adam_epochs}, seed={PLOT_SEED}."
        )
        axes[row_index, 0].set_visible(False)

    if lbfgs_df.empty:
        print(
            f"Warning: no L-BFGS loss history found for "
            f"Adam={adam_epochs}, seed={PLOT_SEED}."
        )
        axes[row_index, 1].set_visible(False)

    for column, label in loss_columns:

        if not adam_df.empty:
            axes[row_index, 0].plot(
                adam_df["step"],
                adam_df[column],
                label=label,
                linewidth=1.7,
            )

        if not lbfgs_df.empty:
            axes[row_index, 1].plot(
                lbfgs_df["step"],
                lbfgs_df[column],
                label=label,
                linewidth=1.7,
            )

    if not adam_df.empty:
        axes[row_index, 0].set_title(
            f"Adam: {adam_epochs:,} epochs"
        )
        axes[row_index, 0].set_ylabel("Loss")
        axes[row_index, 0].set_yscale("log")
        axes[row_index, 0].set_xlim(left=0)
        axes[row_index, 0].grid(
            True,
            which="both",
            alpha=0.25,
        )

    if not lbfgs_df.empty:
        axes[row_index, 1].set_title(
            "L-BFGS refinement"
        )
        axes[row_index, 1].set_yscale("log")
        axes[row_index, 1].set_xlim(left=0)
        axes[row_index, 1].grid(
            True,
            which="both",
            alpha=0.25,
        )

axes[-1, 0].set_xlabel("Adam epoch")
axes[-1, 1].set_xlabel(
    "L-BFGS objective evaluations"
)

# Obtain the legend from the first visible axis
# containing plotted lines.
legend_axis = None

for axis in axes.flat:
    if axis.get_visible() and len(axis.lines) > 0:
        legend_axis = axis
        break

if legend_axis is not None:
    handles, labels = (
        legend_axis.get_legend_handles_labels()
    )

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=4,
        frameon=False,
    )

fig.tight_layout(rect=[0, 0, 1, 0.96])

figure_png = (
    FIGURES_DIR / "adam_lbfgs_loss_histories.png"
)
figure_pdf = (
    FIGURES_DIR / "adam_lbfgs_loss_histories.pdf"
)

fig.savefig(
    figure_png,
    dpi=600,
    bbox_inches="tight",
)

fig.savefig(
    figure_pdf,
    bbox_inches="tight",
)

plt.show()

print("Saved:", figure_png)
print("Saved:", figure_pdf)

In [ ]:
# ============================================================
# Publication-table preview
# ============================================================

table_rows = []

for _, row in summary.iterrows():
    table_rows.append({
        "Adam epochs": int(row["adam_epochs"]),
        "Runs": int(row["runs"]),
        r"Rel. L2(psi)": (
            f"{row['relL2_psi_mean']:.4e} ± "
            f"{row['relL2_psi_std']:.2e}"
        ),
        r"Rel. L2(n)": (
            f"{row['relL2_n_mean']:.4e} ± "
            f"{row['relL2_n_std']:.2e}"
        ),
        r"Rel. L2(I)": (
            f"{row['relL2_I_mean']:.4e} ± "
            f"{row['relL2_I_std']:.2e}"
        ),
        "Aggregate error": (
            f"{row['aggregate_error_mean']:.4e} ± "
            f"{row['aggregate_error_std']:.2e}"
        ),
        "Adam time (min)": (
            f"{row['adam_runtime_s_mean']/60.0:.1f} ± "
            f"{row['adam_runtime_s_std']/60.0:.1f}"
        ),
        "L-BFGS time (min)": (
            f"{row['lbfgs_runtime_s_mean']/60.0:.1f} ± "
            f"{row['lbfgs_runtime_s_std']/60.0:.1f}"
        ),
        "Total time (min)": (
            f"{row['training_runtime_s_mean']/60.0:.1f} ± "
            f"{row['training_runtime_s_std']/60.0:.1f}"
        ),
        "Actual L-BFGS iterations": (
            f"{row['lbfgs_iterations_mean']:.0f} ± "
            f"{row['lbfgs_iterations_std']:.0f}"
        ),
        "L-BFGS objective evaluations": (
            f"{row['lbfgs_objective_evaluations_mean']:.0f} ± "
            f"{row['lbfgs_objective_evaluations_std']:.0f}"
        ),
    })

table_preview = pd.DataFrame(table_rows)
table_preview.to_csv(TABLE_CSV, index=False)

display(table_preview)
print("Saved:", TABLE_CSV)

print(
    "\nSelect the final Adam setting using both accuracy and "
    "total training cost. A larger Adam count should only be "
    "preferred when it gives a meaningful and repeatable "
    "improvement in the three relative L2 errors."
)